# 03 · Ejercicios — Diseños $2^k$ completos
(R)

**Semana 3 — Diseños $2^k$ y fraccionados.**

**Objetivos**
- Analizar diseños $2^3$ con réplicas mediante ANOVA completo e identificar efectos significativos.
- Usar la gráfica de probabilidad normal de efectos en diseños $2^4$ sin réplicas.
- Ajustar modelos reducidos y encontrar las condiciones óptimas del proceso.
- Validar supuestos de normalidad y varianza constante.

**Paquetes:** base R, `car`

> Teoría: [../teoria/01-disenos-2k.md](../teoria/01-disenos-2k.md)  
> Equivalente en Python: [03-ejercicios-2k_py.ipynb](03-ejercicios-2k_py.ipynb)

In [ ]:
suppressMessages(library(car))
options(scipen = 5, digits = 4)

---
## Ejercicio 1 · Rendimiento de reacción química ($2^3$ con réplicas)

Tres factores a dos niveles, dos réplicas (16 corridas). Respuesta: **rendimiento (%)**.

| Factor | $-1$ | $+1$ |
|--------|------|------|
| temperatura (A) | 70 °C | 90 °C |
| tiempo (B) | 30 min | 60 min |
| catalizador (C) | 1 % | 2 % |

In [ ]:
df1 <- read.csv('../datos/reaccion-quimica-2k.csv')
df1$temperatura <- factor(df1$temperatura)
df1$tiempo      <- factor(df1$tiempo)
df1$catalizador <- factor(df1$catalizador)
cat('Dimensiones:', dim(df1), '\n')
print(head(df1))

In [ ]:
# Tabla de medias
print(tapply(df1$rendimiento, list(df1$temperatura, df1$tiempo), mean))

In [ ]:
op <- par(mfrow = c(1, 3))
boxplot(rendimiento ~ temperatura, data = df1, main = 'Por temperatura',
        col = 'lightsteelblue', xlab = 'Temperatura (cod)')
boxplot(rendimiento ~ tiempo, data = df1, main = 'Por tiempo',
        col = 'lightsteelblue', xlab = 'Tiempo (cod)')
boxplot(rendimiento ~ catalizador, data = df1, main = 'Por catalizador',
        col = 'lightsteelblue', xlab = 'Catalizador (cod)')
par(op)

In [ ]:
# Modelo completo 2^3
modelo1 <- aov(rendimiento ~ temperatura * tiempo * catalizador, data = df1)
print(summary(modelo1))

In [ ]:
# Grafica de interaccion temperatura x tiempo
interaction.plot(
    df1$tiempo, df1$temperatura, df1$rendimiento,
    xlab = 'Tiempo', ylab = 'Rendimiento medio (%)',
    main = 'Interaccion Temperatura x Tiempo',
    col = c('steelblue','tomato'), lwd = 2, lty = 1:2,
    legend = TRUE, trace.label = 'Temperatura'
)

In [ ]:
# Modelo reducido: A, B, C, AB
modelo1r <- lm(rendimiento ~ temperatura + tiempo + catalizador + temperatura:tiempo, data = df1)
print(summary(modelo1r))

# Diagnosticos
op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo1r), main = 'Q-Q residuales - reaccion')
qqline(residuals(modelo1r))
plot(fitted(modelo1r), residuals(modelo1r),
     xlab = 'Valores ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados')
abline(h = 0, lty = 2)
par(op)
print(shapiro.test(residuals(modelo1r)))

---
## Ejercicio 2 · Resistencia del papel ($2^4$ sin réplicas)

Cuatro factores, 16 corridas sin réplicas. Respuesta: **resistencia a la traccion (N/m)**.
Se usa la grafica de probabilidad normal de efectos (Daniel) para identificar activos.

Factores: temperatura (A), humedad (B), velocidad (C), presion (D).

In [ ]:
df2 <- read.csv('../datos/papel-resistencia-2k.csv')
cat('Dimensiones:', dim(df2), '\n')
print(head(df2))

# Ajustar modelo saturado con variables numericas (-1/+1)
modelo2 <- lm(resistencia ~ temperatura*humedad*velocidad*presion, data = df2)

# Efectos = 2 * coeficiente
efectos2 <- coef(modelo2)[-1] * 2
nombres2 <- c('A','B','C','D','AB','AC','AD','BC','BD','CD','ABC','ABD','ACD','BCD','ABCD')
names(efectos2) <- nombres2
cat('\nEfectos estimados (por magnitud):\n')
print(sort(abs(efectos2), decreasing = TRUE))

In [ ]:
# Grafica de probabilidad normal de efectos (Daniel)
eff_ord <- sort(efectos2)
n2 <- length(eff_ord)
probs2 <- (seq_len(n2) - 0.5) / n2
z2 <- qnorm(probs2)

plot(z2, eff_ord, pch = 16, col = 'steelblue',
     xlab = 'Cuantiles normales', ylab = 'Efecto estimado',
     main = 'Grafica normal de efectos — Papel resistencia (Daniel)')
abline(h = 0, lty = 2, col = 'gray')
text(z2, eff_ord, names(eff_ord), pos = 4, cex = 0.85)
grid()

In [ ]:
# Modelo reducido: A, B, D, AD
modelo2r <- lm(resistencia ~ temperatura + humedad + presion + temperatura:presion, data = df2)
print(summary(modelo2r))

# Diagnosticos
op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo2r), main = 'Q-Q residuales - papel')
qqline(residuals(modelo2r))
plot(fitted(modelo2r), residuals(modelo2r),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados - papel')
abline(h = 0, lty = 2)
par(op)
print(shapiro.test(residuals(modelo2r)))

# Prediccion optima
opt2 <- data.frame(temperatura = 1, humedad = -1, presion = 1)
cat('\nPrediccion optima (A=+1, B=-1, D=+1):', predict(modelo2r, opt2), 'N/m\n')

---
## Ejercicio 3 · Grabado de semiconductores ($2^3$ con réplicas)

Tres factores, dos réplicas (16 corridas). Respuesta: **tasa de grabado (Å/min)**.

| Factor | $-1$ | $+1$ |
|--------|------|------|
| potencia (A) | 100 W | 150 W |
| presion_gas (B) | 0.8 Torr | 1.2 Torr |
| temperatura (C) | 15 °C | 25 °C |

In [ ]:
df3 <- read.csv('../datos/microelectronica-2k.csv')
df3$potencia     <- factor(df3$potencia)
df3$presion_gas  <- factor(df3$presion_gas)
df3$temperatura  <- factor(df3$temperatura)

op <- par(mfrow = c(1, 3))
boxplot(tasa_grabado ~ potencia, data = df3, main = 'Por potencia',
        col = 'lightgreen', ylab = 'Tasa grabado (A/min)')
boxplot(tasa_grabado ~ presion_gas, data = df3, main = 'Por presion gas',
        col = 'lightgreen')
boxplot(tasa_grabado ~ temperatura, data = df3, main = 'Por temperatura',
        col = 'lightgreen')
par(op)

In [ ]:
modelo3 <- aov(tasa_grabado ~ potencia * presion_gas * temperatura, data = df3)
print(summary(modelo3))

In [ ]:
# Interaccion potencia x temperatura (la mas importante segun ANOVA)
op <- par(mfrow = c(1, 2))
interaction.plot(df3$temperatura, df3$potencia, df3$tasa_grabado,
    xlab = 'Temperatura', ylab = 'Tasa grabado media',
    main = 'Potencia x Temperatura',
    col = c('steelblue','tomato'), lwd = 2, trace.label = 'Potencia')

interaction.plot(df3$potencia, df3$presion_gas, df3$tasa_grabado,
    xlab = 'Potencia', ylab = 'Tasa grabado media',
    main = 'Potencia x Presion gas',
    col = c('steelblue','tomato'), lwd = 2, trace.label = 'Presion')
par(op)

In [ ]:
# Medias por celda — encontrar el optimo
medias3 <- with(df3, tapply(tasa_grabado, list(potencia, presion_gas, temperatura), mean))
cat('Media maxima observada:', max(medias3), '\n')
cat('En combinacion:', which(medias3 == max(medias3), arr.ind = TRUE), '\n')

# Modelo reducido
modelo3r <- lm(tasa_grabado ~ potencia + presion_gas + temperatura + potencia:temperatura,
               data = df3)
cat('\nR2 modelo reducido:', summary(modelo3r)$r.squared, '\n')

op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo3r), main = 'Q-Q residuales - semiconductores')
qqline(residuals(modelo3r))
plot(fitted(modelo3r), residuals(modelo3r),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados')
abline(h = 0, lty = 2)
par(op)
print(shapiro.test(residuals(modelo3r)))

---
## Ejercicio 4 · Densidad de espuma de poliuretano ($2^4$ sin réplicas)

Cuatro factores, 16 corridas sin réplicas. Respuesta: **densidad (kg/m³)**.

Factores: isocianato (A), poliol (B), temperatura (C), humedad (D).

In [ ]:
df4 <- read.csv('../datos/espuma-poliuretano-2k.csv')
cat('Dimensiones:', dim(df4), '\n')

modelo4 <- lm(densidad ~ isocianato*poliol*temperatura*humedad, data = df4)
efectos4 <- coef(modelo4)[-1] * 2
nombres4 <- c('A','B','C','D','AB','AC','AD','BC','BD','CD','ABC','ABD','ACD','BCD','ABCD')
names(efectos4) <- nombres4
cat('\nEfectos (magnitud):\n')
print(round(sort(abs(efectos4), decreasing = TRUE), 2))

In [ ]:
# Grafica de Daniel
eff_ord4 <- sort(efectos4)
n4 <- length(eff_ord4)
z4 <- qnorm((seq_len(n4) - 0.5) / n4)

plot(z4, eff_ord4, pch = 16, col = 'darkorange',
     xlab = 'Cuantiles normales', ylab = 'Efecto estimado',
     main = 'Grafica normal de efectos — Espuma poliuretano (Daniel)')
abline(h = 0, lty = 2, col = 'gray')
text(z4, eff_ord4, names(eff_ord4), pos = 4, cex = 0.85)
grid()

In [ ]:
# Modelo reducido: A, B, C, D, AC
modelo4r <- lm(densidad ~ isocianato + poliol + temperatura + humedad + isocianato:temperatura,
               data = df4)
print(summary(modelo4r))

op <- par(mfrow = c(1, 2))
qqnorm(residuals(modelo4r), main = 'Q-Q residuales - espuma')
qqline(residuals(modelo4r))
plot(fitted(modelo4r), residuals(modelo4r),
     xlab = 'Ajustados', ylab = 'Residuales',
     main = 'Residuales vs. ajustados - espuma')
abline(h = 0, lty = 2)
par(op)
print(shapiro.test(residuals(modelo4r)))

# Prediccion en combinaciones extremas
comb4 <- data.frame(
    isocianato = c(1, 1,-1,-1),
    poliol     = c(-1,-1,-1,-1),
    temperatura = c(-1,-1, 1, 1),
    humedad    = c( 1,-1, 1,-1)
)
comb4$pred <- predict(modelo4r, comb4)
cat('\nPredicciones en combinaciones extremas:\n')
print(comb4)